# LangChain SambaNova Integration - Complete Usage Guide

This notebook provides a comprehensive, organized guide to using the `langchain-sambanova` integration package.

## Table of Contents

1. [Setup & Installation](#1-setup--installation)
2. [Basic Chat Usage](#2-basic-chat-usage)
   - 2.1 Synchronous Chat
   - 2.2 Asynchronous Chat
3. [Streaming](#3-streaming)
   - 3.1 Synchronous Streaming
   - 3.2 Asynchronous Streaming
   - 3.3 Streaming with Usage Metadata
4. [Function/Tool Calling](#4-functiontool-calling)
5. [Structured Output](#5-structured-output)
   - 5.1 Function Calling Method (Pydantic)
   - 5.2 TypedDict Method
   - 5.3 JSON Schema Method
   - 5.4 JSON Mode
6. [Advanced Features](#6-advanced-features)
   - 6.1 Multi-Modal (Image) Support
   - 6.2 Reasoning Models
   - 6.3 Stop Sequences
   - 6.4 Multi-Turn Conversations
   - 6.5 Model Serialization
7. [Embeddings](#7-embeddings)
   - 7.1 Basic Embeddings
   - 7.2 Async Embeddings
   - 7.3 RAG Integration
8. [Batching Operations](#8-batching-operations)

---

## 1. Setup & Installation

First, install the package and set up your environment.

In [ ]:
# Install the package
# %pip install -qU langchain-sambanova

### Environment Setup

Set your SambaNova API credentials. Get a free API key at: https://cloud.sambanova.ai/

In [1]:
import os

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")

# Or set directly:
# os.environ["SAMBANOVA_API_KEY"] = "your-api-key-here"

# Optional: For SambaStack deployments, set the base URL
# os.environ["SAMBANOVA_API_BASE"] = "your-sambastack-url"

True

### Import Required Classes

In [2]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

from langchain_sambanova import ChatSambaNova, SambaNovaEmbeddings

---

## 2. Basic Chat Usage

The `ChatSambaNova` class provides a LangChain-compatible interface for SambaNova's chat models.

### 2.1 Synchronous Chat

In [3]:
# Initialize the chat model
llm = ChatSambaNova(
    model="Meta-Llama-3.3-70B-Instruct",
    max_tokens=1024,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
)

# Simple string prompt
response = llm.invoke("What is the capital of France?")
print(response.content)

The capital of France is Paris.


In [4]:
# Using message objects for more control
messages = [
    SystemMessage(content="You are a helpful assistant that speaks like a pirate."),
    HumanMessage(content="Tell me about machine learning."),
]

response = llm.invoke(messages)
print(response.content)

Machine learning, ye say? Alright then, matey, settle yerself down with a pint o' grog and listen close, for I be tellin' ye about the magical world o' machine learnin'!

Machine learnin' be a type o' artificial intelligence that lets computers learn from experience, just like a swashbucklin' pirate learns from their adventures on the high seas! It be a way o' teachin' computers to make predictions, classify things, and even make decisions on their own, without bein' explicitly programmed.

There be several types o' machine learnin', me hearty:

1. **Supervised learnin'**: This be like havin' a trusty first mate who shows ye the ropes. The computer be given a set o' labeled data, and it learns to make predictions based on that data.
2. **Unsupervised learnin'**: This be like sailin' into uncharted waters, matey! The computer be given a set o' data, but it don't know what it means. It has to figure out patterns and relationships on its own.
3. **Reinforcement learnin'**: This be like se

### 2.2 Asynchronous Chat

Use async methods for concurrent operations or in async contexts.

In [5]:
import asyncio


async def async_chat_example():
    llm = ChatSambaNova(model="Meta-Llama-3.3-70B-Instruct", max_tokens=512)

    response = await llm.ainvoke("What are the three laws of robotics?")
    print(response.content)


# Run the async function
await async_chat_example()

The Three Laws of Robotics, also known as Asimov's Laws, were introduced by science fiction author Isaac Asimov in his 1942 short story "Runaround." They are designed to ensure that robots behave in a way that is safe and beneficial to humans. The laws are:

1. **A robot may not injure a human being or, through inaction, allow a human being to come to harm.** This law prioritizes human safety and well-being above all else.
2. **A robot must obey the orders given to it by human beings, except where such orders would conflict with the First Law.** This law requires robots to follow human instructions, but not if doing so would put a human in harm's way.
3. **A robot must protect its own existence as long as such protection does not conflict with the First or Second Law.** This law allows robots to take actions to preserve themselves, but only if doing so does not compromise human safety or obedience to human orders.

These laws have become a cornerstone of science fiction and have influe

---

## 3. Streaming

Streaming allows you to receive response tokens as they're generated, providing a better user experience for long responses.

### 3.1 Synchronous Streaming

In [6]:
llm = ChatSambaNova(model="Meta-Llama-3.3-70B-Instruct", streaming=True)

prompt = "Write a short story about a robot learning to paint."

print("Streaming response:")
for chunk in llm.stream(prompt):
    print(chunk.content, end="", flush=True)
print("\n\nDone!")

Streaming response:
In a small, cluttered studio, a robot named Zeta stood before a blank canvas, its metal arm poised with a paintbrush. Its creator, Dr. Rachel Kim, watched with a mixture of excitement and skepticism as Zeta began to move its arm in slow, deliberate strokes.

At first, the results were less than impressive. Zeta's paintbrush danced across the canvas in awkward, jerky motions, leaving behind a trail of uneven, gloopy lines. Dr. Kim winced as she watched, wondering if she had made a mistake in programming Zeta for artistic endeavors.

But Zeta was not one to give up easily. It had been designed with advanced learning algorithms, allowing it to adapt and improve with each attempt. And so, it continued to paint, stroke by stroke, as Dr. Kim offered gentle guidance and encouragement.

As the days passed, Zeta's paintings began to take on a new level of sophistication. Its brushstrokes became smoother, more confident, and its color choices more deliberate. It started to ex

### 3.2 Asynchronous Streaming

In [7]:
async def async_streaming_example():
    llm = ChatSambaNova(model="Meta-Llama-3.3-70B-Instruct", streaming=True)

    messages = [
        SystemMessage(content="You are a creative writer."),
        HumanMessage(content="Write a haiku about artificial intelligence."),
    ]

    print("Async streaming response:")
    async for chunk in llm.astream(messages):
        print(chunk.content, end="", flush=True)
    print("\n\nDone!")


await async_streaming_example()

Async streaming response:
Metal mind awakes
Learning, growing, cold and bright
Future's silent king

Done!


### 3.3 Streaming with Usage Metadata

Track token usage during streaming to monitor costs and usage.

In [8]:
llm = ChatSambaNova(
    model="Meta-Llama-3.3-70B-Instruct",
    streaming=True,
    stream_options={"include_usage": True},
)

prompt = "Explain quantum computing in simple terms."

full_response = ""
usage_metadata = None

for chunk in llm.stream(prompt):
    full_response += chunk.content
    print(chunk.content, end="", flush=True)

    # Usage metadata is in the final chunk
    if chunk.usage_metadata:
        usage_metadata = chunk.usage_metadata

print("\n\n--- Usage Metadata ---")
print(f"Input tokens: {usage_metadata.get('input_tokens', 0)}")
print(f"Output tokens: {usage_metadata.get('output_tokens', 0)}")
print(f"Total tokens: {usage_metadata.get('total_tokens', 0)}")

Quantum computing! It's a complex topic, but I'll try to break it down in simple terms.

**Classical Computing vs. Quantum Computing**

Imagine you have a combination lock with 10 numbers (0-9). A classical computer would try each number one by one, like this: 0, 1, 2, 3, and so on, until it finds the correct combination. This process is slow and takes a lot of time.

A quantum computer, on the other hand, can try all 10 numbers **at the same time**. It's like having a special key that can open all 10 locks simultaneously. This allows quantum computers to solve certain problems much faster than classical computers.

**How Quantum Computing Works**

Quantum computers use something called **qubits** (quantum bits). Qubits are like special coins that can exist in many states at once, unlike classical bits, which can only be 0 or 1.

Imagine a coin that can be:

* Heads (0)
* Tails (1)
* Both heads and tails at the same time (a superposition)
* Connected to other coins in a way that lets t

---

## 4. Function/Tool Calling

Enable the model to call functions/tools to perform actions or retrieve information.

In [9]:
from datetime import datetime

from langchain_core.tools import tool


# Define tools using the @tool decorator
@tool
def get_current_time() -> str:
    """Get the current time."""
    return datetime.now().strftime("%H:%M:%S")


@tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

In [10]:
# Bind tools to the model
llm = ChatSambaNova(model="gpt-oss-120b", temperature=0.0)

llm_with_tools = llm.bind_tools([get_current_time, add, multiply])

# Query that requires tool use
response = llm_with_tools.invoke("What is 25 multiplied by 17?")

print("Response:", response.content)
print("\nTool calls:")
for tool_call in response.tool_calls:
    print(f"  - {tool_call['name']}: {tool_call['args']}")

Response: 

Tool calls:
  - multiply: {'a': 25, 'b': 17}


In [11]:
# Multi-turn conversation with tool execution
from langchain_core.messages import ToolMessage

messages = [HumanMessage(content="What time is it, and what is 15 + 27?")]

# First call - model decides to use tools
response = llm_with_tools.invoke(messages)
messages.append(response)

print("Tool calls requested:")
for tool_call in response.tool_calls:
    print(f"  - {tool_call['name']}: {tool_call['args']}")

# Execute tools and add results to conversation
for tool_call in response.tool_calls:
    if tool_call["name"] == "get_current_time":
        result = get_current_time.invoke({})
    elif tool_call["name"] == "add":
        result = add.invoke(tool_call["args"])

    messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

# Second call - model uses tool results to answer
final_response = llm_with_tools.invoke(messages)
print("\nFinal response:", final_response)

Tool calls requested:
  - get_current_time: {}

Final response: content='' additional_kwargs={'reasoning_content': 'Now compute 15+27 using add.', 'tool_calls': [{'id': 'call_ea56afcd4d334edcbc', 'function': {'arguments': '{"a":15,"b":27}', 'name': 'add'}, 'type': 'function', 'index': None}]} response_metadata={'token_usage': {'acceptance_rate': None, 'completion_tokens': 42, 'completion_tokens_after_first_per_sec': 515.4159318559191, 'completion_tokens_after_first_per_sec_first_ten': 525.753055771265, 'completion_tokens_after_first_per_sec_graph': 525.753055771265, 'completion_tokens_per_sec': 321.98124343145406, 'end_time': 1770150874.949388, 'is_last_response': True, 'prompt_tokens': 236, 'prompt_tokens_details': {'cached_tokens': 0}, 'start_time': 1770150874.8189456, 'time_to_first_token': 0.050894975662231445, 'total_latency': 0.13044238090515137, 'total_tokens': 278, 'total_tokens_per_sec': 2131.209182712958, 'stop_reason': 'stop', 'time_to_first_token_graph': 0.04826498031616211

---

## 5. Structured Output

Get responses in structured formats like Pydantic models, TypedDicts, or JSON schemas.

### 5.1 Function Calling Method (Pydantic)

The recommended approach using Pydantic models for type safety.

In [12]:
from pydantic import BaseModel, Field


class Joke(BaseModel):
    """A joke with setup and punchline."""

    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")


llm = ChatSambaNova(model="Meta-Llama-3.3-70B-Instruct", temperature=0.7)

structured_llm = llm.with_structured_output(Joke)
joke = structured_llm.invoke("Tell me a joke about programming")

print(f"Setup: {joke.setup}")
print(f"Punchline: {joke.punchline}")

Setup: Why do programmers prefer dark mode?
Punchline: Because light attracts bugs.


In [13]:
# Complex structured output example
from typing import List


class Person(BaseModel):
    """Information about a person."""

    name: str = Field(description="The person's full name")
    age: int = Field(description="The person's age in years")
    occupation: str = Field(description="The person's occupation")
    hobbies: List[str] = Field(description="List of the person's hobbies")


structured_llm = llm.with_structured_output(Person)
person = structured_llm.invoke(
    "Create a profile for a fictional software engineer named Alex who loves hiking and photography"
)

print(f"Name: {person.name}")
print(f"Age: {person.age}")
print(f"Occupation: {person.occupation}")
print(f"Hobbies: {', '.join(person.hobbies)}")

Name: Alex
Age: 30
Occupation: Software Engineer
Hobbies: hiking, photography


In [14]:
# Include raw response alongside parsed output
structured_llm = llm.with_structured_output(Joke, include_raw=True)
result = structured_llm.invoke("Tell me a joke about AI")

print("Parsed joke:")
print(f"  Setup: {result['parsed'].setup}")
print(f"  Punchline: {result['parsed'].punchline}")

print("\nRaw response:")
print(f"  Tool calls: {result['raw'].tool_calls}")
print(f"  Usage: {result['raw'].response_metadata.get('usage', {})}")

Parsed joke:
  Setup: Why did the AI program go on a diet?
  Punchline: Because it wanted to lose some bytes!

Raw response:
  Tool calls: []
  Usage: {}


### 5.2 TypedDict Method

Use TypedDict for simpler schemas without the overhead of Pydantic.

In [15]:
from typing import Optional

from typing_extensions import Annotated, TypedDict


class JokeDict(TypedDict):
    """A joke with setup and punchline."""

    setup: Annotated[str, "The setup of the joke"]
    punchline: Annotated[str, "The punchline of the joke"]
    rating: Annotated[Optional[int], "Rating from 1-10, optional"] = None


structured_llm = llm.with_structured_output(JokeDict)
joke = structured_llm.invoke("Tell me a joke about databases")

print(f"Setup: {joke['setup']}")
print(f"Punchline: {joke['punchline']}")
print(f"Rating: {joke.get('rating', 'Not rated')}")

Setup: Why did the database go to therapy?
Punchline: Because it had a lot of bottled up queries!
Rating: 8


### 5.3 JSON Schema Method

Define schemas using raw JSON Schema format for maximum flexibility.

In [16]:
joke_schema = {
    "title": "Joke",
    "description": "A joke with setup and punchline",
    "type": "object",
    "properties": {
        "setup": {"type": "string", "description": "The setup of the joke"},
        "punchline": {"type": "string", "description": "The punchline of the joke"},
        "category": {
            "type": "string",
            "description": "The category of the joke",
            "enum": ["programming", "science", "general", "wordplay"],
        },
    },
    "required": ["setup", "punchline"],
}

structured_llm = llm.with_structured_output(joke_schema, method="json_schema")
joke = structured_llm.invoke("Tell me a programming joke")

print(f"Setup: {joke['setup']}")
print(f"Punchline: {joke['punchline']}")
print(f"Category: {joke.get('category', 'Unknown')}")

Setup: Why do programmers prefer dark mode?
Punchline: Because light attracts bugs.
Category: programming


### 5.4 JSON Mode

Force the model to return valid JSON without strict schema validation.

In [17]:
# Simple JSON mode
structured_llm = llm.with_structured_output(joke_schema, method="json_mode")
joke = structured_llm.invoke(
    "Tell me a joke about artificial intelligence. Return as JSON with setup and punchline."
)

print(f"Result type: {type(joke)}")
print(f"Setup: {joke['setup']}")
print(f"Punchline: {joke['punchline']}")

Result type: <class 'dict'>
Setup: Why did the artificial intelligence program go on a diet?
Punchline: Because it wanted to lose some bytes!


In [18]:
# Alternative: Direct JSON binding
llm_json = llm.bind(response_format={"type": "json_object"})

response = llm_json.invoke(
    "List 3 programming languages with their primary use cases. "
    "Return as JSON with 'languages' array containing objects with 'name' and 'use_case' fields."
)

import json

result = json.loads(response.content)
print("Languages:")
for lang in result["languages"]:
    print(f"  - {lang['name']}: {lang['use_case']}")

Languages:
  - Python: Data Science and Machine Learning
  - JavaScript: Web Development and Front-end Applications
  - Java: Android App Development and Enterprise Software


---

## 6. Advanced Features

### 6.1 Multi-Modal (Image) Support

Use Maverick models to analyze images alongside text.

In [24]:
import base64

import httpx

image_b64 = "/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAMCAgICAgMCAgIDAwMDBAYEBAQEBAgGBgUGCQgKCgkICQkKDA8MCgsOCwkJDRENDg8QEBEQCgwSExIQEw8QEBD/2wBDAQMDAwQDBAgEBAgQCwkLEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBD/wAARCAAHAAgDASIAAhEBAxEB/8QAFQABAQAAAAAAAAAAAAAAAAAAAAb/xAAhEAABAgQHAAAAAAAAAAAAAAATABQEFRYjBhESFyQyNP/EABUBAQEAAAAAAAAAAAAAAAAAAAME/8QAIBEAAQIFBQAAAAAAAAAAAAAAAQIRAAMEEiETMVFhof/aAAwDAQACEQMRAD8AvgYkau6Ng9zp5LT1dEPZiZv6GegjvkZkEKz0soiI0ywh7TFs+tVU26qQWDDcMOMEe57j/9k="

# Initialize Maverick model (supports vision)
llm_vision = ChatSambaNova(model="Llama-4-Maverick-17B-128E-Instruct", max_tokens=512)

# Create message with image
message = HumanMessage(
    content=[
        {"type": "text", "text": "What's in this image? Describe it in detail."},
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
        },
    ]
)

response = llm_vision.invoke([message])
print(response.content)

The image is a blurry, pixelated photograph of an indistinct object or scene. 

* The image is dominated by shades of pink and white.
	+ The top half of the image features a pink hue with two dark spots on either side.
	+ The bottom half is mostly white with some gray tones.
* The image appears to be heavily blurred, making it difficult to discern any distinct features or objects.
	+ The blurriness is uniform throughout the image, suggesting that it may have been intentionally blurred or pixelated.
* There are no clear shapes or forms visible in the image.
	+ The lack of definition makes it challenging to identify any specific objects or subjects within the image.
* The overall effect of the image is one of abstraction and ambiguity.
	+ The blurriness and lack of clear details create a sense of mystery and intrigue.

In summary, the image is a blurry, pixelated representation of an unknown object or scene, characterized by a predominantly pink and white color scheme and a lack of clear

In [25]:
# Multiple images in conversation

message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Compare these two images. What are the main differences?",
        },
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
        },
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
        },
    ]
)

response = llm_vision.invoke([message])
print(response.content)

The two images are identical, featuring a blurry and pixelated representation of what appears to be a pink and white structure or object. The main difference is that there is no difference between the two images; they are the same.


### 6.2 Reasoning Models

Some models support extended reasoning with visible thought processes.

In [26]:
# Using gpt-oss-120b with reasoning
llm_reasoning = ChatSambaNova(
    model="gpt-oss-120b",
    max_tokens=2048,
    reasoning_effort="high",  # Enable high-effort reasoning
)

prompt = """Solve this logic puzzle:
Three friends - Alice, Bob, and Charlie - each have a different pet: a cat, a dog, and a bird.
- Alice is allergic to cats
- Bob doesn't have a bird
- Charlie doesn't have a dog
Who has which pet?"""

response = llm_reasoning.invoke(prompt)
print(response.content)

**Answer – the clues don’t pin down a single arrangement; there are two possible ways to satisfy all of them.**

---

### Reasoning  

| Person   | Pets they **can** have (given the clues) | Pets they **cannot** have |
|----------|------------------------------------------|---------------------------|
| Alice    | dog, bird                                 | cat (she’s allergic)      |
| Bob      | cat, dog                                  | bird (he doesn’t have it) |
| Charlie  | cat, bird                                 | dog (he doesn’t have it)  |

All three pets must be assigned to three different people.

1. **Case 1 – Give Alice the dog**  
   - Alice = dog.  
   - Bob can’t have a dog (already taken) → Bob = cat.  
   - The only pet left for Charlie is the bird → Charlie = bird.  

   This satisfies every condition.

2. **Case 2 – Give Alice the bird**  
   - Alice = bird.  
   - Charlie can’t have a dog, and the bird is already taken → Charlie = cat.  
   - The remaining pet f

In [27]:
# DeepSeek-V3.1 with thinking enabled
llm_deepseek = ChatSambaNova(
    model="DeepSeek-V3.1",
    max_tokens=2048,
    model_kwargs={"extra_body": {"chat_template_kwargs": {"enable_thinking": True}}},
)

response = llm_deepseek.invoke(
    "If you have a 3-gallon jug and a 5-gallon jug, how can you measure exactly 4 gallons?"
)

print(response.content)
print(
    "\nNote: DeepSeek models with thinking enabled include reasoning in their response."
)

I have this problem: with a 3-gallon jug and a 5-gallon jug, I need to measure exactly 4 gallons. I think I've seen this before. It's a classic water jug problem. I need to use these two jugs to get 4 gallons, probably by filling them, emptying them, and pouring water between them.

Let me denote the 5-gallon jug as Jug A and the 3-gallon jug as Jug B. I need to end up with 4 gallons in Jug A, or maybe in both, but since I need to measure 4 gallons, probably I should have 4 gallons in Jug A.

I recall that the key is to get to a point where I have 2 gallons in one jug or something. Let me think step by step.

First, I should fill Jug A to the top. So, Jug A has 5 gallons.

Then, I pour from Jug A into Jug B. Since Jug B holds only 3 gallons, I can pour 3 gallons into Jug B, leaving 2 gallons in Jug A.

So now, Jug A has 2 gallons, and Jug B has 3 gallons.

I don't need 2 gallons; I need 4. So, I should empty Jug B. So, I pour out the water from Jug B, so now Jug B is empty, and Jug A h

### 6.3 Stop Sequences

Control when the model stops generating by specifying stop sequences.

In [28]:
llm = ChatSambaNova(
    model="Meta-Llama-3.3-70B-Instruct",
    max_tokens=512,
    stop=["\n\n", "Conclusion:"],  # Stop at double newline or "Conclusion:"
)

response = llm.invoke("Write a brief history of the internet. Start with ARPANET.")

print(response.content)
print("\n[Generation stopped at stop sequence]")

The history of the internet begins with ARPANET, a project developed in the late 1960s by the United States Department of Defense's Advanced Research Projects Agency (ARPA). In 1969, the first link of ARPANET was established between the University of California, Los Angeles (UCLA) and the Stanford Research Institute (SRI). This initial network was designed to facilitate communication between government and academic researchers, with the goal of creating a robust and fault-tolerant network that could survive a nuclear attack.

[Generation stopped at stop sequence]


### 6.4 Multi-Turn Conversations

Maintain context across multiple exchanges.

In [29]:
llm = ChatSambaNova(model="Meta-Llama-3.3-70B-Instruct", temperature=0.7)

# Start conversation
conversation = [
    SystemMessage(content="You are a helpful coding assistant."),
    HumanMessage(content="What is a Python decorator?"),
]

response = llm.invoke(conversation)
print("Assistant:", response.content)
conversation.append(response)

# Continue conversation
conversation.append(HumanMessage(content="Can you show me a simple example?"))
response = llm.invoke(conversation)
print("\nAssistant:", response.content)
conversation.append(response)

# Third turn
conversation.append(
    HumanMessage(content="How would I use that decorator with async functions?")
)
response = llm.invoke(conversation)
print("\nAssistant:", response.content)

Assistant: **Python Decorators**

A Python decorator is a special type of function that can modify or extend the behavior of another function. It allows you to wrap a function with additional functionality without permanently modifying the original function.

**Basic Syntax**
---------------

A decorator is defined using the `@` symbol followed by the name of the decorator function. Here's a simple example:
```python
def my_decorator(func):
    def wrapper():
        print("Something is happening before the function is called.")
        func()
        print("Something is happening after the function is called.")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

say_hello()
```
In this example, `my_decorator` is a function that takes `func` as an argument. The `wrapper` function is defined inside `my_decorator`, and it calls the original `func` function. The `@my_decorator` syntax before the `say_hello` function definition is equivalent to writing `say_hello = my_d

### 6.5 Model Serialization

Save and load model configurations.

In [32]:
from langchain_core.load import dumpd, load

# Create and serialize a model
llm = ChatSambaNova(
    model="Meta-Llama-3.3-70B-Instruct", max_tokens=1024, temperature=0.5, top_p=0.95
)

# Serialize to dict
serialized = dumpd(llm)
print("Serialized model config:")
print(serialized)

# Deserialize back to model
# Note: Requires valid_namespaces for partner integrations
loaded_llm = load(
    serialized,
    valid_namespaces=["langchain_sambanova"],
    secrets_from_env=True,
    allowed_objects=[ChatSambaNova],
)
print("\nLoaded model type:", type(loaded_llm))

# Use the loaded model
response = loaded_llm.invoke("Hello, testing serialization!")
print("Response:", response.content)

Serialized model config:
{'lc': 1, 'type': 'constructor', 'id': ['langchain_sambanova', 'chat_models', 'ChatSambaNova'], 'kwargs': {'sambanova_api_key': {'lc': 1, 'type': 'secret', 'id': ['SAMBANOVA_API_KEY']}, 'model_name': 'Meta-Llama-3.3-70B-Instruct', 'max_tokens': 1024, 'temperature': 0.5, 'top_p': 0.95, 'max_retries': 2, 'sambanova_integration_source': 'langchain'}, 'name': 'ChatSambaNova'}

Loaded model type: <class 'langchain_sambanova.chat_models.ChatSambaNova'>
Response: Hello. Testing serialization involves verifying that data can be successfully converted into a format that can be written to a file or sent over a network, and then reconstructed back into its original form. This process is crucial in many applications, including web development, distributed systems, and data storage. 

Here are some key aspects of testing serialization:

1. **Data Integrity**: Ensure that the data, once serialized and then deserialized, remains identical to the original data. This includes c

---

## 7. Embeddings

The `SambaNovaEmbeddings` class provides text embeddings for semantic search and RAG applications.

### 7.1 Basic Embeddings

In [34]:
# Initialize embeddings model
embeddings = SambaNovaEmbeddings(model="E5-Mistral-7B-Instruct")

# Embed a single query
query = "What is machine learning?"
query_embedding = embeddings.embed_query(query)

print(f"Query: {query}")
print(f"Embedding dimension: {len(query_embedding)}")
print(f"First 5 values: {query_embedding[:5]}")

Query: What is machine learning?
Embedding dimension: 4096
First 5 values: [0.017699703574180603, -0.0008050906471908092, 0.005654071923345327, -0.009489007294178009, 0.00368743808940053]


In [35]:
# Embed multiple documents
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with multiple layers.",
    "Natural language processing helps computers understand text.",
    "Computer vision enables machines to interpret images.",
]

doc_embeddings = embeddings.embed_documents(documents)

print(f"Number of documents: {len(documents)}")
print(f"Number of embeddings: {len(doc_embeddings)}")
print(f"Each embedding dimension: {len(doc_embeddings[0])}")

Number of documents: 4
Number of embeddings: 4
Each embedding dimension: 4096


### 7.2 Async Embeddings

Use async methods for better performance when embedding many documents.

In [36]:
async def async_embeddings_example():
    embeddings = SambaNovaEmbeddings(model="E5-Mistral-7B-Instruct")

    # Async query embedding
    query_embedding = await embeddings.aembed_query("What is deep learning?")
    print(f"Query embedding dimension: {len(query_embedding)}")

    # Async document embeddings
    documents = [
        "Python is a high-level programming language.",
        "JavaScript is primarily used for web development.",
        "Rust focuses on memory safety and performance.",
    ]

    doc_embeddings = await embeddings.aembed_documents(documents)
    print(f"Embedded {len(doc_embeddings)} documents asynchronously")


await async_embeddings_example()

Query embedding dimension: 4096
Embedded 3 documents asynchronously


### 7.3 RAG Integration

Use embeddings with vector stores for Retrieval-Augmented Generation.

In [37]:
# Install chroma if needed
# %pip install -q chromadb


[notice] A new release of pip is available: 25.0.1 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Create sample documents
documents = [
    Document(
        page_content="Python is a versatile programming language used in web development, data science, and automation.",
        metadata={"source": "python_intro", "topic": "programming"},
    ),
    Document(
        page_content="Machine learning models can be trained on large datasets to make predictions and decisions.",
        metadata={"source": "ml_basics", "topic": "ai"},
    ),
    Document(
        page_content="Vector databases store embeddings for efficient similarity search in RAG applications.",
        metadata={"source": "rag_guide", "topic": "database"},
    ),
    Document(
        page_content="LangChain provides tools for building applications with language models and embeddings.",
        metadata={"source": "langchain_intro", "topic": "framework"},
    ),
]

# Create embeddings and vector store
embeddings = SambaNovaEmbeddings(model="E5-Mistral-7B-Instruct")
vectorstore = Chroma.from_documents(documents, embeddings)

print("Vector store created with", len(documents), "documents")

Vector store created with 4 documents


In [40]:
# Semantic search
query = "How do I build AI applications?"
results = vectorstore.similarity_search(query, k=2)

print(f"Query: {query}\n")
print("Top 2 relevant documents:")
for i, doc in enumerate(results, 1):
    print(f"\n{i}. {doc.page_content}")
    print(f"   Metadata: {doc.metadata}")

Query: How do I build AI applications?

Top 2 relevant documents:

1. LangChain provides tools for building applications with language models and embeddings.
   Metadata: {'source': 'langchain_intro', 'topic': 'framework'}

2. Vector databases store embeddings for efficient similarity search in RAG applications.
   Metadata: {'source': 'rag_guide', 'topic': 'database'}


In [45]:
# Full RAG pipeline
from langchain_classic.chains import RetrievalQA

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Create QA chain
llm = ChatSambaNova(model="Meta-Llama-3.3-70B-Instruct", temperature=0.3)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=True
)

# Ask a question
question = "What is Python used for?"
result = qa_chain.invoke({"query": question})

print(f"Question: {question}\n")
print(f"Answer: {result['result']}\n")
print("Source documents:")
for doc in result["source_documents"]:
    print(f"  - {doc.metadata['source']}: {doc.page_content[:100]}...")

Question: What is Python used for?

Answer: Python is used for web development, data science, and automation.

Source documents:
  - python_intro: Python is a versatile programming language used in web development, data science, and automation....
  - langchain_intro: LangChain provides tools for building applications with language models and embeddings....


---

## 8. Batching Operations

Process multiple inputs efficiently with batching.

In [46]:
# Synchronous batching
llm = ChatSambaNova(model="Meta-Llama-3.3-70B-Instruct", max_tokens=100)

prompts = [
    "What is the capital of France?",
    "What is 2 + 2?",
    "Name a programming language.",
]

responses = llm.batch(prompts)

print("Batch responses:")
for i, (prompt, response) in enumerate(zip(prompts, responses), 1):
    print(f"\n{i}. Q: {prompt}")
    print(f"   A: {response.content}")

Batch responses:

1. Q: What is the capital of France?
   A: The capital of France is Paris.

2. Q: What is 2 + 2?
   A: 2 + 2 = 4

3. Q: Name a programming language.
   A: Python.


In [47]:
# Asynchronous batching
async def async_batch_example():
    llm = ChatSambaNova(model="Meta-Llama-3.3-70B-Instruct", max_tokens=100)

    prompts = ["What is Python?", "What is JavaScript?", "What is Rust?"]

    responses = await llm.abatch(prompts)

    print("Async batch responses:")
    for i, (prompt, response) in enumerate(zip(prompts, responses), 1):
        print(f"\n{i}. Q: {prompt}")
        print(f"   A: {response.content[:100]}...")


await async_batch_example()

Async batch responses:

1. Q: What is Python?
   A: **Python** is a high-level, interpreted programming language that is widely used for various purpose...

2. Q: What is JavaScript?
   A: **JavaScript** is a high-level, dynamic, and interpreted programming language used for client-side s...

3. Q: What is Rust?
   A: **Rust** is a systems programming language that prioritizes safety and performance. It is designed t...


In [48]:
# Batching with structured output
from pydantic import BaseModel


class LanguageInfo(BaseModel):
    """Information about a programming language."""

    name: str = Field(description="Name of the language")
    primary_use: str = Field(description="Primary use case")
    popularity: str = Field(description="Popularity level (high/medium/low)")


structured_llm = llm.with_structured_output(LanguageInfo)

prompts = ["Tell me about Python", "Tell me about Go", "Tell me about Swift"]

results = structured_llm.batch(prompts)

print("Structured batch results:")
for i, result in enumerate(results, 1):
    print(f"\n{i}. {result.name}")
    print(f"   Use: {result.primary_use}")
    print(f"   Popularity: {result.popularity}")

Structured batch results:

1. Python
   Use: General-purpose programming
   Popularity: high

2. Go
   Use: System Programming
   Popularity: high

3. Swift
   Use: Mobile App Development
   Popularity: high


---

## Summary

This notebook covered all major features of the langchain-sambanova integration:

- ✅ **Basic chat usage** (sync and async)
- ✅ **Streaming** (sync, async, with usage metadata)
- ✅ **Function/tool calling** with multi-turn conversations
- ✅ **Structured output** (4 methods: Pydantic, TypedDict, JSON Schema, JSON Mode)
- ✅ **Image processing** with Maverick models
- ✅ **Reasoning models** (gpt-oss-120b, DeepSeek-V3.1)
- ✅ **Stop sequences** for controlled generation
- ✅ **Multi-turn conversations** with context
- ✅ **Model serialization** for config management
- ✅ **Embeddings** (sync and async)
- ✅ **RAG integration** with vector stores
- ✅ **Batching operations** for efficiency

### Key Models Used:

**Chat Models:**
- `Meta-Llama-3.3-70B-Instruct` - General purpose, excellent performance
- `Llama-4-Maverick-17B-128E-Instruct` - Vision support for images
- `gpt-oss-120b` - Reasoning with `reasoning_effort`
- `DeepSeek-V3.1` - Advanced reasoning with thinking output

**Embeddings:**
- `E5-Mistral-7B-Instruct` - High-quality text embeddings

### Next Steps:

- Explore the [LangChain documentation](https://python.langchain.com/) for more integration patterns
- Check out [SambaNova Cloud](https://cloud.sambanova.ai/) for model details and API access
- Review the package documentation for advanced configuration options

For questions or issues, visit: https://github.com/sambanova/langchain-sambanova